In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
interpretation_log = []

def evaluate_and_interpret(name, model, X_test, y_test, best_params=None, cv_score=None):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {'model': name, 'accuracy': acc, 'precision': prec,
               'recall': rec, 'f1': f1, 'auc': auc}
    results.append(metrics)

    recall_quality = "strong" if rec > 0.7 else "moderate" if rec > 0.5 else "weak"
    auc_quality = "strong" if auc > 0.8 else "moderate" if auc > 0.65 else "weak"

    md = f"""### {name}

**Best Hyperparameters (5-fold Stratified CV, optimized for recall):** {best_params if best_params else 'N/A'}
**CV Recall Score:** {f'{cv_score:.3f}' if cv_score else 'N/A'}

**Test Set Metrics:**
- Accuracy: {acc:.3f}
- Precision: {prec:.3f}
- Recall (Sensitivity): {rec:.3f}
- F1 Score: {f1:.3f}
- AUC: {auc:.3f}

**Confusion Matrix:** TN={tn}, FP={fp}, FN={fn}, TP={tp}

**Interpretation:**
- Correctly identifies {rec*100:.1f}% of truly anemic women (recall) — {recall_quality} for this health screening context.
- Of women predicted anemic, {prec*100:.1f}% actually are (precision).
- AUC of {auc:.3f} indicates {auc_quality} discriminative ability.
- False negatives (missed anemia cases): {fn}.
"""
    display(Markdown(md))
    interpretation_log.append(md)
    return metrics

In [10]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [1, 3, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    rf_param_grid,
    cv=skf,
    scoring='recall',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_

evaluate_and_interpret(
    'Random Forest',
    best_rf,
    X_test,
    y_test,
    best_params=rf_grid.best_params_,
    cv_score=rf_grid.best_score_
)

TypeError: evaluate_and_interpret() got an unexpected keyword argument 'best_params'